# Import

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm
from ucimlrepo import fetch_ucirepo 
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load and Scale Dataset

## Load

In [22]:
  
# fetch dataset 
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 
  
# metadata 
print(breast_cancer_wisconsin_diagnostic.metadata) 
  
# variable information 
print(breast_cancer_wisconsin_diagnostic.variables) 


{'uci_id': 17, 'name': 'Breast Cancer Wisconsin (Diagnostic)', 'repository_url': 'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic', 'data_url': 'https://archive.ics.uci.edu/static/public/17/data.csv', 'abstract': 'Diagnostic Wisconsin Breast Cancer Database.', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 569, 'num_features': 30, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Diagnosis'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1993, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C5DW2B', 'creators': ['William Wolberg', 'Olvi Mangasarian', 'Nick Street', 'W. Street'], 'intro_paper': {'ID': 230, 'type': 'NATIVE', 'title': 'Nuclear feature extraction for breast tumor diagnosis', 'authors': 'W. Street, W. Wolberg, O. Mangasarian', 'venue': 'Electronic imaging', 'year': 1993, 'journal': None, 'DOI': '1

In [23]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

In [24]:
y.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Diagnosis  569 non-null    str  
dtypes: str(1)
memory usage: 4.6 KB


# Label Encoding

In [25]:
y

,Diagnosis
0,M
1,M
2,M
3,M
4,M
...,...
564,M
565,M
566,M
567,M


In [26]:
le = LabelEncoder()
y = pd.Series(
    le.fit_transform(y),
    index=y.index,
    name="Severity"
)
y.value_counts()
y

c:\Users\chris\Documents\GitHub\ML_DL\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0      1
1      1
2      1
3      1
4      1
      ..
564    1
565    1
566    1
567    1
568    0
Name: Severity, Length: 569, dtype: int64

## Split

In [27]:
X_train, X_test_val, y_train, y_test_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_test, X_val, y_test, y_val = train_test_split(
    X_test_val,
    y_test_val,
    test_size=0.5,   # 0.2 of 0.8 = 0.16 of the full dataset
    random_state=42,
    stratify=y_test_val
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 455
Validation size: 57
Test size: 57


## Standardize

In [28]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


# Dataloader
A DataLoader is used to load the dataset in small batches during training. It can also shuffle the training data, which helps the model learn better. In PyTorch, it is the standard way to iterate over data efficiently and cleanly.

In [29]:
# Convert data to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)



y_train = torch.tensor(y_train.squeeze().to_numpy(), dtype=torch.long)
y_val = torch.tensor(y_val.squeeze().to_numpy(), dtype=torch.long)
y_test = torch.tensor(y_test.squeeze().to_numpy(), dtype=torch.long)


# Create datasets and dataloaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 4

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [30]:
next(iter(train_loader))

[tensor([[ 1.3717,  1.2581,  1.4615,  1.2344, -0.3626,  2.1386,  1.4890,  1.2763,
           1.7926,  0.1027,  1.9661,  0.8122,  2.7907,  1.3164, -0.1218,  4.2759,
           2.0575,  2.3382,  3.9636,  1.4758,  1.5807,  1.1963,  2.0370,  1.3072,
          -0.3148,  3.0897,  2.1198,  2.0040,  2.7890,  1.0842],
         [-0.2478, -1.3294, -0.2603, -0.3271, -0.8649, -0.3358, -0.4707, -0.5140,
          -0.7169, -0.9730, -0.7862, -1.2454, -0.6222, -0.5384, -0.9203, -0.2609,
          -0.1179, -0.5130, -0.7007, -0.4386, -0.4313, -1.4054, -0.3301, -0.4588,
          -0.6601,  0.0820,  0.0540, -0.3559, -0.2821, -0.6019],
         [ 0.7479,  0.0099,  0.6555,  0.6128, -1.5071, -0.5869, -0.4618, -0.5374,
           0.1051, -1.4412,  0.2336,  1.5264,  0.2676,  0.1103, -1.2107,  0.0731,
          -0.0460, -0.0940, -0.6296, -0.8547,  0.3817,  0.3565,  0.3644,  0.2431,
          -1.8905, -0.5342, -0.4105, -0.4662, -0.6140, -1.3447],
         [-0.1416, -0.0671, -0.1116, -0.2341, -0.3794,  0.2018,  0.

# Model

In [31]:
class base_MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        
        # A simple network with one hidden layer
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [32]:
class advanced_MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()

        # A simple network with one hidden layer and dropout
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [33]:
base_model = base_MLP(30, 32, 2)
advanced_model = advanced_MLP(30, 32, 2, 0.3)

In [34]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
base_model_optimizer = optim.Adam(base_model.parameters(), lr=0.01)
advanced_model_optimzer = optim.Adam(advanced_model.parameters(),lr=0.01)

# Training

## Base model

In [35]:
num_epochs = 20

for epoch in range(num_epochs):
    # -----------------------------
    # Training
    # -----------------------------
    base_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch_X, batch_y in progress_bar:
        # Forward pass
        train_outputs = base_model(batch_X)
        train_loss = criterion(train_outputs, batch_y)

        # Backward pass and parameter update
        base_model_optimizer.zero_grad()
        train_loss.backward()
        base_model_optimizer.step()

        # Collect training statistics
        train_loss_sum += train_loss.item() * batch_X.size(0)
        train_predictions = torch.argmax(train_outputs, dim=1)
        train_correct += (train_predictions == batch_y).sum().item()
        train_total += batch_y.size(0)

        progress_bar.set_postfix({
            "batch_loss": f"{train_loss.item():.4f}"
        })

    train_loss_epoch = train_loss_sum / train_total
    train_acc_epoch = train_correct / train_total

    # -----------------------------
    # Validation
    # -----------------------------
    base_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            val_outputs = base_model(batch_X)
            val_loss = criterion(val_outputs, batch_y)

            val_loss_sum += val_loss.item() * batch_X.size(0)
            val_predictions = torch.argmax(val_outputs, dim=1)
            val_correct += (val_predictions == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_loss_epoch = val_loss_sum / val_total
    val_acc_epoch = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss_epoch:.4f} | "
        f"Train Acc: {train_acc_epoch:.4f} | "
        f"Val Loss: {val_loss_epoch:.4f} | "
        f"Val Acc: {val_acc_epoch:.4f}"
    )




Epoch [1/20] | Train Loss: 0.1279 | Train Acc: 0.9495 | Val Loss: 0.0204 | Val Acc: 1.0000


Epoch [2/20] | Train Loss: 0.0739 | Train Acc: 0.9736 | Val Loss: 0.0405 | Val Acc: 0.9649


Epoch [3/20] | Train Loss: 0.0532 | Train Acc: 0.9824 | Val Loss: 0.0136 | Val Acc: 1.0000


Epoch [4/20] | Train Loss: 0.0430 | Train Acc: 0.9846 | Val Loss: 0.0049 | Val Acc: 1.0000


Epoch [5/20] | Train Loss: 0.0887 | Train Acc: 0.9736 | Val Loss: 0.0564 | Val Acc: 0.9825


...

Epoch [16/20] | Train Loss: 0.0065 | Train Acc: 0.9978 | Val Loss: 0.0003 | Val Acc: 1.0000


Epoch [17/20] | Train Loss: 0.0031 | Train Acc: 1.0000 | Val Loss: 0.0002 | Val Acc: 1.0000


Epoch [18/20] | Train Loss: 0.0020 | Train Acc: 1.0000 | Val Loss: 0.0002 | Val Acc: 1.0000


Epoch [19/20] | Train Loss: 0.0023 | Train Acc: 1.0000 | Val Loss: 0.0002 | Val Acc: 1.0000


Epoch [20/20] | Train Loss: 0.0021 | Train Acc: 1.0000 | Val Loss: 0.0002 | Val Acc: 1.0000


In [36]:

# Final evaluation on the test set
base_model.eval()

with torch.no_grad():
    test_outputs = base_model(X_test)
    test_predictions = torch.argmax(test_outputs, dim=1)
    test_accuracy = (test_predictions == y_test).float().mean()

print("\nFinal Test Accuracy:", test_accuracy.item())

# Convert tensors to numpy arrays for scikit-learn
y_true = y_test.numpy()
y_pred = test_predictions.numpy()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=breast_cancer_wisconsin_diagnostic.target_names))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Final Test Accuracy: 0.8947368264198303

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92        36
           1       0.89      0.81      0.85        21

    accuracy                           0.89        57
   macro avg       0.89      0.88      0.88        57
weighted avg       0.89      0.89      0.89        57

Confusion Matrix:
[[34  2]
 [ 4 17]]


## Model with Dropout and Early stopping

In [37]:

num_epochs = 100
patience = 10  # Stop if validation accuracy does not improve for 10 epochs


In [38]:

best_val_accuracy = 0.0
best_model_state = {
    key: value.detach().clone()
    for key, value in advanced_model.state_dict().items()
}
epochs_without_improvement = 0

for epoch in range(num_epochs):
    # -----------------------------
    # Training
    # -----------------------------
    advanced_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch_X, batch_y in train_progress:
        # Forward pass
        train_outputs = advanced_model(batch_X)
        loss = criterion(train_outputs, batch_y)

        # Backward pass and parameter update
        advanced_model_optimzer.zero_grad()
        loss.backward()
        advanced_model_optimzer.step()

        # Collect training statistics
        train_loss_sum += loss.item() * batch_X.size(0)
        predictions = torch.argmax(train_outputs, dim=1)
        train_correct += (predictions == batch_y).sum().item()
        train_total += batch_y.size(0)

        train_progress.set_postfix({
            "batch_loss": f"{loss.item():.4f}"
        })

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # -----------------------------
    # Validation
    # -----------------------------
    advanced_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            val_outputs = advanced_model(batch_X)
            val_loss = criterion(val_outputs, batch_y)

            val_loss_sum += loss.item() * batch_X.size(0)
            predictions = torch.argmax(val_outputs, dim=1)
            val_correct += (predictions == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # -----------------------------
    # Early stopping
    # -----------------------------
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_state = {
            key: value.detach().clone()
            for key, value in advanced_model.state_dict().items()
        }
        epochs_without_improvement = 0
        print("Validation accuracy improved, saving model state.")
        print(f"Patience: {epochs_without_improvement}/{patience}")
    else:
        epochs_without_improvement += 1
        print(f"Patience: {epochs_without_improvement}/{patience}")

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break



Epoch [1/100] | Train Loss: 0.1690 | Train Acc: 0.9385 | Val Loss: 0.0065 | Val Acc: 0.9825
Validation accuracy improved, saving model state.
Patience: 0/10


Epoch [2/100] | Train Loss: 0.0845 | Train Acc: 0.9736 | Val Loss: 0.0000 | Val Acc: 1.0000
Validation accuracy improved, saving model state.
Patience: 0/10


Epoch [3/100] | Train Loss: 0.0654 | Train Acc: 0.9736 | Val Loss: 0.0000 | Val Acc: 1.0000
Patience: 1/10


Epoch [4/100] | Train Loss: 0.0810 | Train Acc: 0.9626 | Val Loss: 0.0000 | Val Acc: 1.0000
Patience: 2/10


Epoch [5/100] | Train Loss: 0.1023 | Train Acc: 0.9670 | Val Loss: 0.0017 | Val Acc: 0.9825
Patience: 3/10


...

Epoch [8/100] | Train Loss: 0.0420 | Train Acc: 0.9824 | Val Loss: 0.0631 | Val Acc: 1.0000
Patience: 6/10


Epoch [9/100] | Train Loss: 0.0433 | Train Acc: 0.9890 | Val Loss: 0.0000 | Val Acc: 1.0000
Patience: 7/10


Epoch [10/100] | Train Loss: 0.1024 | Train Acc: 0.9802 | Val Loss: 0.0016 | Val Acc: 0.9825
Patience: 8/10


Epoch [11/100] | Train Loss: 0.0943 | Train Acc: 0.9868 | Val Loss: 0.0184 | Val Acc: 1.0000
Patience: 9/10


Epoch [12/100] | Train Loss: 0.0361 | Train Acc: 0.9868 | Val Loss: 0.0000 | Val Acc: 1.0000
Patience: 10/10

Early stopping triggered.


In [39]:

# Load the best model
advanced_model.load_state_dict(best_model_state)

advanced_model.eval()
test_correct = 0
test_total = 0

all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        test_outputs = advanced_model(batch_X)
        test_predictions = torch.argmax(test_outputs, dim=1)

        # Update accuracy statistics
        test_correct += (test_predictions == batch_y).sum().item()
        test_total += batch_y.size(0)

        # Store predictions and true labels
        all_y_true.extend(batch_y.numpy())
        all_y_pred.extend(test_predictions.numpy())

test_accuracy = test_correct / test_total
print("\nFinal Test Accuracy:", test_accuracy)

# Print classification report
print("\nClassification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=breast_cancer_wisconsin_diagnostic.target_names))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(all_y_true, all_y_pred))


Final Test Accuracy: 0.9473684210526315

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96        36
           1       0.95      0.90      0.93        21

    accuracy                           0.95        57
   macro avg       0.95      0.94      0.94        57
weighted avg       0.95      0.95      0.95        57

Confusion Matrix:
[[35  1]
 [ 2 19]]
